In [1]:
from google.cloud import bigquery

client = bigquery.Client(project="prod-organize-arizon-4e1c0a83")

query = """
SELECT
*
FROM `prod-organize-arizon-4e1c0a83.viewers_dataset.cd7_2024_raul`
"""

df = client.query(query).to_dataframe()

df.head()

,PCTNUM,GEOMETRY,dem_votes,rep_votes,third_votes,total_votes
0,PM0005,"POLYGON((-111.794981 32.506644, -111.795082 32...",1029,2444,6,3479
1,PN0038,"POLYGON((-111.877606 32.872217, -111.877573 32...",196,250,0,446
2,PN0067,"POLYGON((-112.203548 32.836227, -112.203942 32...",489,1269,2,1760
3,PN0091,"POLYGON((-111.722806 32.762925, -111.74058 32....",34,96,1,131
4,PM0215,"POLYGON((-111.218155 32.313572, -111.184011 32...",541,734,0,1275


In [2]:

# have to create geometry that folium can work with from bigquery friendly wkt geometry field
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.validation import make_valid  # Shapely ≥2.0

# point to the column that contains WKT, here it's called GEOMETRY
wkt_col = "GEOMETRY"

# parse WKT into shapely geometries
geoms = gpd.GeoSeries.from_wkt(df[wkt_col])

# build a GeoDataFrame and assign the correct CRS
# If WKT is already lon/lat (WGS84), use EPSG:4326. Otherwise, set the true source EPSG and then .to_crs(4326).
gdf = gpd.GeoDataFrame(df.drop(columns=["geometry"], errors="ignore"),
                       geometry=geoms,
                       crs="EPSG:4326")


In [11]:
import folium
import numpy as np
import pandas as pd
from branca.colormap import LinearColormap
from folium.features import GeoJson, GeoJsonTooltip

# ============================================
KEY_COL  = "PCTNUM"
DEM_COL  = "dem_votes"
REP_COL  = "rep_votes"
OTH_COL  = "third_votes"
PCT_NAME = "PRECINCTNA"
# ============================================

# Re-compute Dem margin as TWO-PARTY share centered at 0: (D - R) / (D + R)
# This gives values in [-1, +1]; blue = negative (more Dem), red = positive (more Rep).
twoparty = gdf[[DEM_COL, REP_COL]].sum(axis=1, skipna=True)
with np.errstate(divide='ignore', invalid='ignore'):
    gdf["dem_margin_2p"] = np.where(twoparty > 0,
                                    (gdf[REP_COL] - gdf[DEM_COL]) / twoparty,
                                    np.nan)


minx, miny, maxx, maxy = gdf.total_bounds
m = folium.Map(location=[(miny+maxy)/2, (minx+maxx)/2], zoom_start=6, tiles="CartoDB Positron")

abs_margin = gdf["dem_margin_2p"].abs().dropna()
vmax = float(np.quantile(abs_margin, 0.98)) if len(abs_margin) else 1.0
cmap = LinearColormap(colors=["#2166ac", "#f7f7f7", "#b2182b"], vmin=-vmax, vmax=vmax)
cmap.caption = "Rep margin (two-party, Dem − Rep)"
cmap.add_to(m)


def style_fn(feat):
    val = feat["properties"].get("dem_margin_2p")
    if pd.isna(val):
        return {"fillColor": "#cccccc", "fillOpacity": 0.25, "weight": 0.4, "color": "#666"}
    return {"fillColor": cmap(val), "fillOpacity": 0.8, "weight": 0.4, "color": "#666"}

tooltip_fields = [c for c in [PCT_NAME, "dem_margin_2p"] if c in gdf.columns]
tooltip_aliases = ["Precinct", "party margin (2-party):"]

poly_layer = GeoJson(
    data=gdf.to_json(),
    name="Precincts — Rep margin",
    style_function=style_fn,
    highlight_function=lambda f: {"weight": 2, "color": "#000"},
    tooltip=GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases, localize=True, sticky=False),
)
poly_layer.add_to(m)


folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m.save("az_cd7_2024_results_pctnum.html")


from IPython.display import IFrame
IFrame("map.html", width="100%", height=600)

m

AssertionError: fields and aliases must have the same length.